In [4]:
# Import required libraries
import pandas as pd
import json
import os
from pathlib import Path

In [8]:
# Set up project paths - discover root directory and create data folder
# Dynamically find project root by looking for a marker file/folder
# Works regardless of who runs it or where the repo is cloned
NOTEBOOK_DIR = Path.cwd()

# Walk up until we find the project root (identified by pyproject.toml, .git, or src folder)
PROJECT_ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (parent / '.git').exists() or (parent / 'pyproject.toml').exists():
        PROJECT_ROOT = parent
        break

DATA_DIR = PROJECT_ROOT / 'data'
DATA_OUTPUTS_DIR = PROJECT_ROOT / 'data_outputs' / 'round2' 
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / 'queries').mkdir(exist_ok=True)

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")

Project root: c:\Users\tremopoulosa\OneDrive - Reed Elsevier Group ICO Reed Elsevier Inc\Documents\code\writing-coach-eval
Data dir: c:\Users\tremopoulosa\OneDrive - Reed Elsevier Group ICO Reed Elsevier Inc\Documents\code\writing-coach-eval\data


In [3]:
full_df = pd.read_csv(DATA_DIR / 'data_routes_expanded.csv')
full_df.shape

(208, 5)

In [22]:
respond_results_path = DATA_OUTPUTS_DIR / 'respond_only' / 'data_routes_expanded_RESPOND_details.jsonl'
with open(respond_results_path, 'r') as f:
    respond_results_json = [json.loads(line) for line in f if line.strip()]

research_results_path = DATA_OUTPUTS_DIR / 'research_only' / 'data_routes_expanded_RESEARCH_details.jsonl'
with open(research_results_path, 'r') as f:
    research_results_json = [json.loads(line) for line in f if line.strip()]

revise_research_results_path = DATA_OUTPUTS_DIR / 'revise_research_only' / 'data_routes_expanded_REVISE_RESEARCH_details.jsonl'
with open(revise_research_results_path, 'r') as f:
    revise_research_results_json = [json.loads(line) for line in f if line.strip()]

revise_simple_results_path = DATA_OUTPUTS_DIR / 'revise_simple_only' / 'data_routes_expanded_REVISE_SIMPLE_details.jsonl'
with open(revise_simple_results_path, 'r') as f:
    revise_simple_results_json = [json.loads(line) for line in f if line.strip()]

In [23]:
# Reload clean copies to avoid stale state from re-runs
df_responds = pd.json_normalize(respond_results_json)
print(df_responds.shape)

df_researchs = pd.json_normalize(research_results_json)
print(df_researchs.shape)

df_rev_researchs = pd.json_normalize(revise_research_results_json)
print(df_rev_researchs.shape)

df_rev_simples = pd.json_normalize(revise_simple_results_json)
print(df_rev_simples.shape)

(52, 12)
(52, 12)
(52, 12)
(52, 12)


In [28]:
# input_preview typically ends with "..." — strip it before matching
# Also normalize leading/trailing whitespace on both sides
def find_full_input(row):
    preview = row['input_preview']
    query = row['query']
    # Remove trailing ellipsis and strip whitespace
    prefix = preview.rstrip('.').strip()
    
    query_matches = full_df[full_df['query'] == query]
    
    # Strategy 1: startswith after stripping whitespace on both sides
    for _, frow in query_matches.iterrows():
        if frow['input'].strip().startswith(prefix):
            return frow['input']
    
    # Strategy 2: the preview (without ...) IS the full input (short texts)
    for _, frow in query_matches.iterrows():
        if frow['input'].strip() == preview.strip():
            return frow['input']
    
    return None

In [43]:
df_researchs['input'] = df_researchs.apply(find_full_input, axis=1)

mask1 = df_researchs['input_preview'].str.startswith('Prior work has utilized human feedback as a re')
df_researchs.loc[mask1, 'input'] = "Prior work has utilized human feedback as a reward to train models in numerous domains of Natural Language Processing (NLP). These domains include dialogue, translation, semantic parsing, story generation, review generation, and evidence extraction [2]. Human feedback has also been used in the development of reward modeling approaches, specifically in learning to rank, which has been employed in ranking search results using either explicit or implicit feedback [2]. Another line of research has used human feedback to train agents in simulated environments [2]. Furthermore, research in NLP has extensively used Reinforcement Learning (RL) to optimize automatic metrics such as ROUGE for summarization and BLEU for translation [2]. Studies have also been conducted to improve the performance of Machine Translation (MT) models with minimal human feedback [0]. In extractive Question Answering (QA) systems, explicit feedback from users has been used to continually improve system performance [9]."

mask2 = df_researchs['input_preview'].str.startswith('The OmniTab and TAPEX models propose different')
df_researchs.loc[mask2, 'input'] = "The OmniTab and TAPEX models propose different techniques for table question answering using neural NLP models. OmniTab is an omnivorous pretraining approach that uses both natural and synthetic data to train models for table-based question answering [3]. It first uses synthetic questions to simulate real table-based questions, which involve various reasoning operations such as comparative operations. The synthetic questions are created by generating complex structured queries such as SQL, which are then converted into natural language (NL) questions using an SQL2NL model [3]. The resultant NL questions are used in pretraining directly, with a standard generative QA loss that takes NL questions concatenated with tables as input and decodes answers obtained by executing SQL queries over tables [3]. The synthetic NL questions can help close the gap between SQL and NL questions, especially when limited annotated NL questions are available [3]. On the other hand, TAPEX uses a table pre-training method that learns to execute Neural SQL [5]. In fine-tuning, it feeds the concatenation of an NL sentence and its corresponding table from the downstream task to the model, and trains it to output the answer [5]. TAPEX models both table-based question answering (TableQA) and table-based fact verification (TableFV) as sequence generation tasks, and leverages generative language models to generate the output autoregressively. For example, given an NL question, the method generates the answer by decoding it in a word-by-word fashion [5]."

# print(df_researchs[df_researchs['input'].isna()][['input_preview', 'query']])

matched = df_researchs['input'].notna().sum()
print(f"Matched: {matched} / {len(df_researchs)}")
if matched < len(df_researchs):
    print("Unmatched rows:")
    print(df_researchs[df_researchs['input'].isna()][['input_preview', 'query']])
df_researchs[['input_preview', 'input', 'query']].head(2)

Matched: 52 / 52


,input_preview,input,query
0,Tensor decomposition techniques can be used fo...,Tensor decomposition techniques can be used fo...,"Retrieve ""Tensorizing Neural Networks"" and sum..."
1,"Tensor decomposition techniques, such as the T...","Tensor decomposition techniques, such as the T...","Retrieve ""Compressing deep neural networks by ..."


In [30]:
df_responds['input'] = df_responds.apply(find_full_input, axis=1)

matched = df_responds['input'].notna().sum()
print(f"Matched: {matched} / {len(df_responds)}")
if matched < len(df_responds):
    print("Unmatched rows:")
    print(df_responds[df_responds['input'].isna()][['input_preview', 'query']])
df_responds[['input_preview', 'input', 'query']].head(2)

Matched: 52 / 52


,input_preview,input,query
0,Previous approaches to improve the faithfulnes...,Previous approaches to improve the faithfulnes...,What are the strong results that these models ...
1,Prior approaches to improve the faithfulness o...,Prior approaches to improve the faithfulness o...,Which specific tasks have these models been ap...


In [44]:
df_rev_researchs['input'] = df_rev_researchs.apply(find_full_input, axis=1)

mask1 = (
    df_rev_researchs['input_preview'].str.startswith('Diffusion models can be applied to domains wit') & 
    df_rev_researchs['query'].str.startswith('Include more methods that have been tried, ma')
)
df_rev_researchs.loc[mask1, 'input'] = "Diffusion models can be applied to domains with discrete output spaces, such as text, by using a more structured categorical corruption process to shape data generation [5]. This process involves introducing specific distortions or 'corruptions' to the data during the generative process. For instance, one method within this approach uses a class of transition matrices that introduce local, ordinal inductive biases for structured data like text. Banddiagonal transition matrices can be used, which allow the corruption process to transition only between adjacent states, biasing the reverse process towards local iterative refinement [5]. This results in a more controlled and structured data generation, making it possible to apply diffusion models to text, images, and other domains with discrete output spaces.\n Another method used involves the insertion of [MASK] tokens, drawing parallels to autoregressive and mask-based generative models [5]. Some researchers have attempted to represent each token as a continuous embedding and apply diffusion in the embedding space. However, this method has limitations such as requiring training a customized attribute classifier, as the diffusion operates on a learned embedding space [2].\n A different strategy includes encoding discrete or categorical data into bits, then modeling these bits as real numbers, a technique referred to as using ""analog bits"". This method enables continuous state diffusion models to generate discrete data [1]. Techniques like self-conditioning, where diffusion models are conditioned directly on their previously generated samples, and asymmetric time intervals have been proposed to improve the quality of generated samples [1]. \n However, it's important to note that despite these advancements, diffusion models for text still generally underperform when compared to autoregressive language models [2]."

mask2 = (
    df_rev_researchs['input_preview'].str.startswith('In-context learning is an emergent ability of') & 
    df_rev_researchs['query'].str.startswith('Can you provide more evidence for why in-conte')
)
df_rev_researchs.loc[mask2, 'input'] = "In-context learning is an emergent ability of large language models (LMs) that allows them to learn new tasks or concepts without any parameter updates [2]. It works by conditioning the LM on a text-based prompt that contains input-output pairs relevant to the task at hand. The prompt serves as the context for the task and provides the necessary information for the LM to make accurate predictions.\n When trained on massive text corpora, LMs like GPT-3 implicitly learn to infer latent concepts shared across sentences in a document [0]. These latent concepts can be thought of as representations of knowledge or patterns in the data. During pretraining, the LM learns to generate coherent continuations by inferring these latent concepts across multiple sentences.\n The mechanics of in-context learning involve providing the LM with a prompt that concatenates independent ""training"" examples followed by a ""test example"" [0]. The LM is conditioned on this prompt, and it must infer the shared prompt concept across examples to make a prediction. By leveraging the previously learned latent concepts that capture the long-range coherence in the pretraining data, the LM can effectively perform in-context learning.\n The ability of LMs to perform in-context learning is an emergent property that arises from their ability to model the latent concepts in the data. By inferring the task from the examples in the prompt, the LM can make predictions based on the learned knowledge and patterns.\nIt is important to note that in-context learning is not explicitly pretrained in LMs like GPT-3. The distribution of prompts, which concatenates independent examples, is quite different from natural language. However, the LM's ability to infer the shared prompt concept across examples allows it to perform in-context learning [0]. \nIn summary, in-context learning is enabled by the LM's ability to leverage previously learned latent concepts to infer the task from examples in the prompt. It is an emergent ability that arises from the LM's training on massive text corpora and its ability to capture long-range coherence in the data [2][0]."

# print(df_rev_researchs[df_rev_researchs['input'].isna()][['input_preview', 'query']])

matched = df_rev_researchs['input'].notna().sum()
print(f"Matched: {matched} / {len(df_rev_researchs)}")
if matched < len(df_rev_researchs):
    print("Unmatched rows:")
    print(df_rev_researchs[df_rev_researchs['input'].isna()][['input_preview', 'query']])
df_rev_researchs[['input_preview', 'input', 'query']].head(2)

Matched: 52 / 52


,input_preview,input,query
0,Contrastive learning techniques have been appl...,Contrastive learning techniques have been appl...,"In the first couple of sentences, provide a ge..."
1,Diffusion models can be applied to domains wit...,Diffusion models can be applied to domains wit...,"Include more methods that have been tried, may..."


In [32]:
df_rev_simples['input'] = df_rev_simples.apply(find_full_input, axis=1)

matched = df_rev_simples['input'].notna().sum()
print(f"Matched: {matched} / {len(df_rev_simples)}")
if matched < len(df_rev_simples):
    print("Unmatched rows:")
    print(df_rev_simples[df_rev_simples['input'].isna()][['input_preview', 'query']])
df_rev_simples[['input_preview', 'input', 'query']].head(2)

Matched: 52 / 52


,input_preview,input,query
0,In distributed systems the consistency of data...,In distributed systems the consistency of data...,Fix fluency in this text
1,Cognition-based factors have been utilized as ...,Cognition-based factors have been utilized as ...,Rewrite the answer to be concise and directly ...


In [46]:
# Define output paths
output_base = "data_outputs/whole_input"
paths = {
    "respond": f"{output_base}/respond_only/data_routes_expanded_RESPOND_results",
    "research": f"{output_base}/research_only/data_routes_expanded_RESEARCH_results",
    "rev_research": f"{output_base}/revise_research_only/data_routes_expanded_REVISE_RESEARCH_results",
    "rev_simple": f"{output_base}/revise_simple_only/data_routes_expanded_REVISE_SIMPLE_results"
}

# Create directories if they don't exist
for path in paths.values():
    Path(path).parent.mkdir(parents=True, exist_ok=True)

# Save as CSV
df_responds.to_csv(f"{paths['respond']}.csv", index=False)
df_researchs.to_csv(f"{paths['research']}.csv", index=False)
df_rev_researchs.to_csv(f"{paths['rev_research']}.csv", index=False)
df_rev_simples.to_csv(f"{paths['rev_simple']}.csv", index=False)

# Save as JSONL
df_responds.to_json(f"{paths['respond']}.jsonl", orient="records", lines=True)
df_researchs.to_json(f"{paths['research']}.jsonl", orient="records", lines=True)
df_rev_researchs.to_json(f"{paths['rev_research']}.jsonl", orient="records", lines=True)
df_rev_simples.to_json(f"{paths['rev_simple']}.jsonl", orient="records", lines=True)